# ChildWhisper — Complete Training & Evaluation Pipeline

**Pasketti Children's ASR Challenge (Word Track)**

Two-model ensemble: Whisper-large-v3 LoRA + Whisper-small full fine-tune.
Target WER: <0.20 | Budget: $0-10/month | GPU: Kaggle T4 / Colab T4 / A100 inference

---

### Table of Contents

| # | Section | Description |
|---|---------|-------------|
| 1 | **Setup & Environment** | Install deps, detect Kaggle/local, configure paths |
| 2 | **Data Upload** | Google Drive → Kaggle dataset transfer (Colab only) |
| 3 | **Data Verification & EDA** | Validate data, duration stats, age distribution |
| 4 | **Whisper-small Training** | Full fine-tune (~6 hrs on T4) |
| 5 | **Whisper-large-v3 LoRA Training** | INT8 + LoRA (~8 hrs on T4) |
| 6 | **Augmented Retraining** | Noise augmentation with RealClass + MUSAN |
| 7 | **AutoWhisper Experiments** | Scripted/agent hyperparameter search |
| 8 | **Evaluation & Visualization** | WER analysis, per-age breakdown, error analysis |
| 9 | **Ensemble Inference & Submission** | Two-model inference + output generation |

> **Usage:** Run sections selectively based on what you need. Each section is self-contained after Section 1.

---
## 1. Setup & Environment

In [ ]:
import subprocess
import sys
import os

# Install dependencies not pre-installed on Kaggle/Colab
deps = [
    "jiwer>=3.0",
    "audiomentations>=0.36",
    "peft>=0.12",
    "bitsandbytes>=0.43",
    "accelerate>=0.33",
]
for dep in deps:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", dep])

print("Dependencies installed.")

In [ ]:
import sys
import os
import time
import json
from pathlib import Path

import numpy as np
import torch
import yaml
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({
    "figure.figsize": (12, 6),
    "figure.dpi": 100,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

# Add project root to path
# Try current dir first, then parent (if running from notebooks/), then Kaggle
PROJECT_ROOT = None
for candidate in [Path(".").resolve(), Path("..").resolve()]:
    if (candidate / "src").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is not None:
    sys.path.insert(0, str(PROJECT_ROOT))
elif Path("/kaggle/input/childwhisper-src/src").exists():
    sys.path.insert(0, "/kaggle/input/childwhisper-src")
    PROJECT_ROOT = Path("/kaggle/input/childwhisper-src")
else:
    raise RuntimeError(
        "Cannot find 'src/' directory. If running locally, start the notebook "
        "from the project root or the notebooks/ directory. If on Kaggle, "
        "attach the 'childwhisper-src' dataset containing your src/ folder."
    )

from src.kaggle_utils import (
    get_paths, get_paths_lora, setup_hub_auth,
    get_latest_checkpoint, download_checkpoint,
    get_kaggle_training_args, get_lora_training_args,
    verify_kaggle_data, check_gpu_memory, is_kaggle,
    unify_kaggle_audio, get_kaggle_noise_paths,
)
from src.preprocess import load_audio, load_metadata, trim_silence, is_silence, get_duration
from src.utils import normalize_text, normalize_and_postprocess
from src.evaluate import (
    compute_wer, compute_per_age_wer, validation_summary,
    compute_error_breakdown, compute_per_age_error_breakdown,
    detect_hallucinations, error_analysis_summary,
    format_error_analysis_report, format_validation_report,
    combined_validation_summary,
)
from src.dataset import create_train_val_split

print(f"Running on Kaggle: {is_kaggle()}")
print(f"Project root: {PROJECT_ROOT}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    print("Device: Apple Silicon (MPS)")
else:
    print("Device: CPU")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CONFIGURATION — Edit these values for your environment
# ══════════════════════════════════════════════════════════════

DATASET_SLUG = "pasketti-audio"             # Kaggle dataset slug
LOCAL_DATA_DIR = "data"                      # Local data directory

# HuggingFace Hub model IDs
SMALL_HUB_ID = "nishantgaurav23/pasketti-whisper-small"
LORA_HUB_ID = "nishantgaurav23/pasketti-whisper-lora"
SMALL_AUG_HUB_ID = "nishantgaurav23/pasketti-whisper-small-augmented"
LORA_AUG_HUB_ID = "nishantgaurav23/pasketti-whisper-lora-augmented"
AUTOWHISPER_HUB_ID = "nishantgaurav23/autowhisper-results"

# Which stages to run (set True/False)
RUN_UPLOAD = False          # Section 2: Data upload (Colab only)
RUN_SMALL_TRAIN = True      # Section 4: Whisper-small training
RUN_LORA_TRAIN = True       # Section 5: Whisper-large-v3 LoRA
RUN_AUGMENTED = False       # Section 6: Augmented retraining
RUN_AUTOWHISPER = False     # Section 7: AutoWhisper experiments

PUSH_TO_HUB = True          # Push checkpoints to HF Hub

# Paths — auto-detects Kaggle vs local, unifies split audio dirs on Kaggle
# Only call get_paths once; derive LoRA output dir without re-running unification
paths = get_paths(dataset_slug=DATASET_SLUG, local_data_dir=LOCAL_DATA_DIR)
AUDIO_DIR = paths["audio_dir"]
METADATA_PATH = paths["metadata_path"]
SMALL_OUTPUT_DIR = paths["output_dir"]

# Reuse paths, just override output_dir for LoRA (avoids redundant unification)
LORA_OUTPUT_DIR = (
    Path("/kaggle/working/checkpoints/whisper-large-v3-lora")
    if is_kaggle()
    else Path("./checkpoints/whisper-large-v3-lora")
)

# Noise directories (for augmented training)
# On Kaggle: auto-detected from pasketti-audio dataset (noise_part_*, musan/)
# Locally: falls back to data/ subdirectories
if is_kaggle():
    NOISE_DIR = paths.get("noise_dir") or Path("/kaggle/input/musan-noise/noise")
    MUSAN_DIR = paths.get("musan_dir")
    REALCLASS_DIR = NOISE_DIR  # RealClass noise unified into noise_dir
else:
    NOISE_DIR = Path("data/musan_noise")
    MUSAN_DIR = Path("data/musan")
    REALCLASS_DIR = Path("data/realclass_noise")

CONFIG_PATH = PROJECT_ROOT / "configs" / "training_config.yaml"
if not CONFIG_PATH.exists():
    # On Kaggle, check the source dataset
    for _candidate in [
        Path("/kaggle/input/childwhisper-src/configs/training_config.yaml"),
        Path("configs/training_config.yaml"),
    ]:
        if _candidate.exists():
            CONFIG_PATH = _candidate
            break

print(f"Audio dir:     {AUDIO_DIR}")
print(f"Metadata:      {METADATA_PATH}")
print(f"Small output:  {SMALL_OUTPUT_DIR}")
print(f"LoRA output:   {LORA_OUTPUT_DIR}")
print(f"Noise dir:     {NOISE_DIR}")
print(f"MUSAN dir:     {MUSAN_DIR}")
print(f"Config:        {CONFIG_PATH}")

In [ ]:
# HuggingFace Hub authentication
# Use env var directly + skip slow token validation and git-credential writes
import os as _os
_hf_token = _os.environ.get("HF_TOKEN")
if _hf_token:
    _os.environ["HUGGING_FACE_HUB_TOKEN"] = _hf_token
    try:
        from huggingface_hub import login as _hf_login
        _hf_login(token=_hf_token, add_to_git_credential=False, new_session=False)
        print("HF Hub authenticated successfully")
    except Exception as e:
        print(f"HF login call failed (token still set via env): {e}")
else:
    print("HF_TOKEN not set — checkpoints will be saved locally only")
    PUSH_TO_HUB = False

---
## 2. Data Upload (Google Drive → Kaggle)

> **Skip this section** if your data is already on Kaggle. This is for initial data transfer from Google Drive via Colab.

In [ ]:
if RUN_UPLOAD:
    # Mount Google Drive (Colab only)
    from google.colab import drive
    drive.mount('/content/drive')

    GDRIVE_FOLDER = "/content/drive/MyDrive/childwhisper/data"
    KAGGLE_USERNAME = "nishantgaurav23"
    KAGGLE_DATASET_SLUG = f"{KAGGLE_USERNAME}/pasketti-audio"
    STAGING_DIR = Path("/content/kaggle_dataset")
    STAGING_DIR.mkdir(parents=True, exist_ok=True)

    gdrive = Path(GDRIVE_FOLDER)
    if gdrive.exists():
        print("Files in Google Drive folder:")
        for f in sorted(gdrive.iterdir()):
            size_mb = f.stat().st_size / 1024 / 1024 if f.is_file() else 0
            print(f"  {f.name:40s} {size_mb:>8.1f} MB" if f.is_file() else f"  {f.name:40s} [DIR]")
    else:
        print(f"ERROR: Folder not found: {GDRIVE_FOLDER}")
else:
    print("Skipping data upload (RUN_UPLOAD=False)")

In [ ]:
if RUN_UPLOAD:
    import shutil
    import zipfile
    import tarfile

    # Setup Kaggle API
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kaggle"])
    KAGGLE_JSON = "/content/drive/MyDrive/kaggle.json"
    if Path(KAGGLE_JSON).exists():
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        shutil.copy2(KAGGLE_JSON, os.path.expanduser("~/.kaggle/kaggle.json"))
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
        print("Kaggle credentials loaded from Google Drive")

    # Copy metadata files
    print("=== Copying metadata files ===")
    for jsonl in gdrive.glob("*.jsonl"):
        dest = STAGING_DIR / jsonl.name
        if not dest.exists():
            shutil.copy2(jsonl, dest)
            print(f"  Copied {jsonl.name}")

    # Extract audio zips
    print("\n=== Extracting audio archives ===")
    audio_dir = STAGING_DIR / "audio"
    audio_dir.mkdir(exist_ok=True)

    for archive in sorted(gdrive.glob("audio*.zip")):
        print(f"  Extracting {archive.name} ({archive.stat().st_size / 1024**3:.1f} GB)...")
        t0 = time.time()
        with zipfile.ZipFile(archive, 'r') as zf:
            zf.extractall(STAGING_DIR)
        print(f"    Done in {time.time() - t0:.0f}s")

    # Upload to Kaggle
    dataset_meta = {
        "title": "Pasketti Children ASR Audio",
        "id": KAGGLE_DATASET_SLUG,
        "licenses": [{"name": "CC-BY-4.0"}]
    }
    (STAGING_DIR / "dataset-metadata.json").write_text(json.dumps(dataset_meta, indent=2))

    result = subprocess.run(
        ["kaggle", "datasets", "create", "-p", str(STAGING_DIR), "--dir-mode", "zip"],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print("Dataset created successfully!")
    elif "already exists" in (result.stderr + result.stdout).lower():
        print("Dataset exists, creating new version...")
        subprocess.run(
            ["kaggle", "datasets", "version", "-p", str(STAGING_DIR),
             "-m", "Updated from Google Drive", "--dir-mode", "zip"],
            capture_output=True, text=True
        )
    else:
        print(f"Error: {result.stderr}")

---
## 3. Data Verification & EDA

In [ ]:
# Verify data availability
stats = verify_kaggle_data(str(AUDIO_DIR), str(METADATA_PATH))

print(f"Utterances:       {stats['num_utterances']}")
print(f"Audio found:      {stats['num_audio_found']}")
print(f"Audio missing:    {stats['num_missing_audio']}")

if stats["duration_stats"]:
    ds = stats["duration_stats"]
    print(f"\nDuration stats:")
    print(f"  Min:          {ds['min']:.2f}s")
    print(f"  Max:          {ds['max']:.2f}s")
    print(f"  Mean:         {ds['mean']:.2f}s")
    print(f"  Total hours:  {ds['total_hours']:.1f}h")

if stats["num_missing_audio"] > 0:
    pct = stats["num_missing_audio"] / stats["num_utterances"] * 100
    print(f"\nWARNING: {pct:.1f}% of audio files are missing!")

In [ ]:
# Load metadata for EDA
metadata = load_metadata(str(METADATA_PATH))
print(f"Loaded {len(metadata)} entries")
if metadata:
    print(f"Keys: {list(metadata[0].keys())}")
    print(f"\nSample entry:")
    print(json.dumps(metadata[0], indent=2))

In [ ]:
# ══════════════════════════════════════════════════════════
# EDA Visualization: Duration Distribution
# ══════════════════════════════════════════════════════════
durations = [m.get("audio_duration_sec", 0) for m in metadata if m.get("audio_duration_sec")]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Duration histogram
axes[0].hist(durations, bins=50, color="steelblue", edgecolor="white", alpha=0.8)
axes[0].axvline(0.3, color="red", linestyle="--", label="Min (0.3s)")
axes[0].axvline(30.0, color="red", linestyle="--", label="Max (30s)")
axes[0].set_xlabel("Duration (seconds)")
axes[0].set_ylabel("Count")
axes[0].set_title("Audio Duration Distribution")
axes[0].legend()

# Duration box plot by age bucket
age_buckets = sorted(set(m.get("age_bucket", "unknown") for m in metadata))
bucket_durations = []
for bucket in age_buckets:
    bucket_durations.append(
        [m["audio_duration_sec"] for m in metadata
         if m.get("age_bucket") == bucket and m.get("audio_duration_sec")]
    )

bp = axes[1].boxplot(bucket_durations, labels=age_buckets, patch_artist=True)
colors = ["#4C72B0", "#55A868", "#C44E52", "#8172B2", "#CCB974"]
for patch, color in zip(bp["boxes"], colors[:len(age_buckets)]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_xlabel("Age Bucket")
axes[1].set_ylabel("Duration (seconds)")
axes[1].set_title("Duration by Age Group")

plt.tight_layout()
plt.savefig("eda_duration_distribution.png", bbox_inches="tight")
plt.show()
print("Saved: eda_duration_distribution.png")

In [ ]:
# ══════════════════════════════════════════════════════════
# EDA Visualization: Age & Speaker Distribution
# ══════════════════════════════════════════════════════════
from collections import Counter

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Age bucket distribution
age_counts = Counter(m.get("age_bucket", "unknown") for m in metadata)
buckets_sorted = sorted(age_counts.keys())
counts = [age_counts[b] for b in buckets_sorted]
bars = axes[0].bar(buckets_sorted, counts, color=colors[:len(buckets_sorted)], alpha=0.8)
for bar, count in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                 str(count), ha='center', va='bottom', fontsize=10)
axes[0].set_xlabel("Age Bucket")
axes[0].set_ylabel("Utterances")
axes[0].set_title("Utterances per Age Group")

# Unique children per age bucket
children_per_bucket = {}
for m in metadata:
    bucket = m.get("age_bucket", "unknown")
    child = m.get("child_id", "unknown")
    children_per_bucket.setdefault(bucket, set()).add(child)
child_counts = [len(children_per_bucket.get(b, set())) for b in buckets_sorted]
bars2 = axes[1].bar(buckets_sorted, child_counts, color=colors[:len(buckets_sorted)], alpha=0.8)
for bar, count in zip(bars2, child_counts):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 str(count), ha='center', va='bottom', fontsize=10)
axes[1].set_xlabel("Age Bucket")
axes[1].set_ylabel("Unique Children")
axes[1].set_title("Unique Speakers per Age Group")

# Transcript length distribution (word count)
word_counts = [len(m.get("orthographic_text", "").split()) for m in metadata]
axes[2].hist(word_counts, bins=range(0, max(word_counts) + 2), color="#55A868",
             edgecolor="white", alpha=0.8)
axes[2].set_xlabel("Word Count")
axes[2].set_ylabel("Utterances")
axes[2].set_title("Transcript Length Distribution (Word Track)")

plt.tight_layout()
plt.savefig("eda_distributions.png", bbox_inches="tight")
plt.show()
print("Saved: eda_distributions.png")

In [ ]:
# ══════════════════════════════════════════════════════════
# EDA: Train/Val Split Visualization
# ══════════════════════════════════════════════════════════
train_meta, val_meta = create_train_val_split(metadata)
print(f"Train: {len(train_meta)} utterances")
print(f"Val:   {len(val_meta)} utterances")
print(f"Ratio: {len(val_meta) / len(metadata):.2%}")

# Verify no speaker leakage
train_children = set(m["child_id"] for m in train_meta)
val_children = set(m["child_id"] for m in val_meta)
overlap = train_children & val_children
print(f"\nTrain children: {len(train_children)}")
print(f"Val children:   {len(val_children)}")
print(f"Overlap:        {len(overlap)} {'(LEAK!)' if overlap else '(clean)'}")

# Split distribution by age
fig, ax = plt.subplots(figsize=(10, 5))
train_ages = Counter(m.get("age_bucket", "unknown") for m in train_meta)
val_ages = Counter(m.get("age_bucket", "unknown") for m in val_meta)

x = np.arange(len(buckets_sorted))
width = 0.35
ax.bar(x - width/2, [train_ages.get(b, 0) for b in buckets_sorted],
       width, label="Train", color="#4C72B0", alpha=0.8)
ax.bar(x + width/2, [val_ages.get(b, 0) for b in buckets_sorted],
       width, label="Val", color="#C44E52", alpha=0.8)
ax.set_xlabel("Age Bucket")
ax.set_ylabel("Utterances")
ax.set_title("Train/Val Split by Age Bucket (child_id stratified, no leakage)")
ax.set_xticks(x)
ax.set_xticklabels(buckets_sorted)
ax.legend()

plt.tight_layout()
plt.savefig("eda_train_val_split.png", bbox_inches="tight")
plt.show()
print("Saved: eda_train_val_split.png")

In [ ]:
# ══════════════════════════════════════════════════════════
# EDA: Sample Audio Waveforms
# ══════════════════════════════════════════════════════════
import librosa
import librosa.display

# Pick up to 4 samples across age buckets
samples = []
seen_buckets = set()
for m in metadata:
    bucket = m.get("age_bucket", "unknown")
    if bucket not in seen_buckets:
        audio_path = AUDIO_DIR / m["audio_path"]
        if audio_path.exists():
            samples.append(m)
            seen_buckets.add(bucket)
    if len(samples) >= 4:
        break

if samples:
    fig, axes = plt.subplots(len(samples), 2, figsize=(16, 3 * len(samples)))
    if len(samples) == 1:
        axes = [axes]

    for i, m in enumerate(samples):
        audio_path = AUDIO_DIR / m["audio_path"]
        audio, sr = load_audio(str(audio_path))
        transcript = m.get("orthographic_text", "")
        bucket = m.get("age_bucket", "?")

        # Waveform
        librosa.display.waveshow(audio, sr=sr, ax=axes[i][0], color="steelblue")
        axes[i][0].set_title(f"[{bucket}] \"{transcript}\" ({get_duration(audio, sr):.2f}s)")
        axes[i][0].set_xlabel("Time (s)")

        # Mel spectrogram
        S = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=80)
        S_dB = librosa.power_to_db(S, ref=np.max)
        librosa.display.specshow(S_dB, sr=sr, x_axis="time", y_axis="mel",
                                 ax=axes[i][1], cmap="magma")
        axes[i][1].set_title(f"Mel Spectrogram")

    plt.tight_layout()
    plt.savefig("eda_audio_samples.png", bbox_inches="tight")
    plt.show()
    print("Saved: eda_audio_samples.png")
else:
    print("No audio files found for visualization.")

---
## 4. Whisper-small Full Fine-Tune

- **Model:** `openai/whisper-small` (242M params)
- **Training:** Full fine-tune with fp16, gradient checkpointing, SpecAugment
- **Expected:** ~6 hrs on T4, WER 0.15-0.20

In [ ]:
if RUN_SMALL_TRAIN:
    with open(CONFIG_PATH) as f:
        config = yaml.safe_load(f)

    ws = config.get("whisper_small", {})
    common = config.get("common", {})

    print("=== Whisper-small Training Config ===")
    print(f"Model:            {ws.get('model_name', 'openai/whisper-small')}")
    print(f"Learning rate:    {ws.get('learning_rate', 1e-5)}")
    print(f"Epochs:           {ws.get('num_train_epochs', 3)}")
    print(f"Batch size:       {ws.get('per_device_train_batch_size', 2)}")
    print(f"Grad accum:       {ws.get('gradient_accumulation_steps', 8)}")
    eff = ws.get('per_device_train_batch_size', 2) * ws.get('gradient_accumulation_steps', 8)
    print(f"Effective batch:  {eff}")
    print(f"SpecAugment:      {common.get('spec_augment', {}).get('apply', True)}")
else:
    print("Skipping Whisper-small training (RUN_SMALL_TRAIN=False)")

In [ ]:
if RUN_SMALL_TRAIN:
    # Check for existing checkpoint to resume
    RESUME_FROM = None
    if PUSH_TO_HUB and get_latest_checkpoint(SMALL_HUB_ID):
        print(f"Found existing checkpoint at {SMALL_HUB_ID}")
        resume_dir = str(SMALL_OUTPUT_DIR / "resume")
        downloaded = download_checkpoint(SMALL_HUB_ID, resume_dir)
        if downloaded:
            RESUME_FROM = downloaded
            print(f"Resuming from {RESUME_FROM}")
    else:
        print("Training from scratch")

In [ ]:
if RUN_SMALL_TRAIN:
    from src.train_whisper_small import main as train_small_main

    train_args = get_kaggle_training_args(
        config_path=str(CONFIG_PATH),
        metadata_path=str(METADATA_PATH),
        audio_dir=str(AUDIO_DIR),
        output_dir=str(SMALL_OUTPUT_DIR),
        resume_from=RESUME_FROM,
    )

    print(f"Starting Whisper-small training...")
    start_time = time.time()
    small_val_wer = train_small_main(train_args)
    small_elapsed = time.time() - start_time

    print(f"\nWhisper-small training complete!")
    print(f"Validation WER: {small_val_wer:.4f}")
    print(f"Elapsed: {small_elapsed / 60:.1f} min ({small_elapsed / 3600:.1f} hrs)")

    # Upload to Hub
    if PUSH_TO_HUB:
        from huggingface_hub import HfApi
        api = HfApi()
        api.upload_folder(
            folder_path=str(SMALL_OUTPUT_DIR),
            repo_id=SMALL_HUB_ID,
            repo_type="model",
            create_pr=False,
        )
        print(f"Uploaded to https://huggingface.co/{SMALL_HUB_ID}")

---
## 5. Whisper-large-v3 LoRA Training

- **Model:** `openai/whisper-large-v3` (1.55B params, 15M trainable via LoRA)
- **Quantization:** INT8 via bitsandbytes (fits T4 16GB)
- **LoRA:** r=32, alpha=64, targets q_proj + v_proj
- **Expected:** ~8 hrs on T4, WER 0.12-0.17

In [ ]:
if RUN_LORA_TRAIN:
    with open(CONFIG_PATH) as f:
        config = yaml.safe_load(f)

    wl = config.get("whisper_large_v3", {})
    lora = wl.get("lora", {})

    print("=== Whisper-large-v3 LoRA Training Config ===")
    print(f"Model:            {wl.get('model_name', 'openai/whisper-large-v3')}")
    print(f"Learning rate:    {wl.get('learning_rate', 1e-4)}")
    print(f"INT8 quant:       {wl.get('load_in_8bit', True)}")
    print(f"LoRA r:           {lora.get('r', 32)}")
    print(f"LoRA alpha:       {lora.get('alpha', 64)}")
    print(f"LoRA targets:     {lora.get('target_modules', ['q_proj', 'v_proj'])}")

    # GPU check
    gpu_info = check_gpu_memory()
    print(f"\nGPU: {gpu_info['gpu_name'] or 'None'}")
    print(f"Memory: {gpu_info['total_memory_gb']:.1f} GB")
    print(f"Sufficient: {gpu_info['is_sufficient']}")
else:
    print("Skipping LoRA training (RUN_LORA_TRAIN=False)")

In [ ]:
if RUN_LORA_TRAIN:
    # Check for existing adapter
    LORA_RESUME = None
    if PUSH_TO_HUB and get_latest_checkpoint(LORA_HUB_ID):
        print(f"Found existing adapter at {LORA_HUB_ID}")
        resume_dir = str(LORA_OUTPUT_DIR / "resume")
        downloaded = download_checkpoint(LORA_HUB_ID, resume_dir)
        if downloaded:
            LORA_RESUME = downloaded
            print(f"Resuming from {LORA_RESUME}")
    else:
        print("Training from scratch")

In [ ]:
if RUN_LORA_TRAIN:
    from src.train_whisper_lora import main as train_lora_main

    lora_args = get_lora_training_args(
        config_path=str(CONFIG_PATH),
        metadata_path=str(METADATA_PATH),
        audio_dir=str(AUDIO_DIR),
        output_dir=str(LORA_OUTPUT_DIR),
        resume_from=LORA_RESUME,
    )

    print(f"Starting LoRA training...")
    start_time = time.time()
    lora_val_wer = train_lora_main(lora_args)
    lora_elapsed = time.time() - start_time

    print(f"\nLoRA training complete!")
    print(f"Validation WER: {lora_val_wer:.4f}")
    print(f"Elapsed: {lora_elapsed / 60:.1f} min ({lora_elapsed / 3600:.1f} hrs)")

    # Upload adapter to Hub
    if PUSH_TO_HUB:
        from huggingface_hub import HfApi
        api = HfApi()
        api.upload_folder(
            folder_path=str(LORA_OUTPUT_DIR),
            repo_id=LORA_HUB_ID,
            repo_type="model",
            create_pr=False,
        )
        print(f"Adapter uploaded to https://huggingface.co/{LORA_HUB_ID}")

---
## 6. Augmented Retraining

Re-trains both models on noise-augmented data:
- 50% RealClass classroom noise (SNR 5-20 dB)
- 20% MUSAN babble noise (SNR 0-15 dB)
- 30% clean audio

**Expected:** ~16 hrs total (8 LoRA + 6 small + overhead)

In [ ]:
if RUN_AUGMENTED:
    # Verify noise directories — files may be .wav or .flac
    noise_files = []
    if NOISE_DIR and NOISE_DIR.exists():
        noise_files = list(NOISE_DIR.glob("*.wav")) + list(NOISE_DIR.glob("*.flac"))
    realclass_files = []
    if REALCLASS_DIR and REALCLASS_DIR.exists():
        realclass_files = list(REALCLASS_DIR.glob("*.wav")) + list(REALCLASS_DIR.glob("*.flac"))
    print(f"Noise files:      {len(noise_files)} in {NOISE_DIR}")
    print(f"RealClass files:  {len(realclass_files)} in {REALCLASS_DIR}")
    assert NOISE_DIR and NOISE_DIR.exists() and len(noise_files) > 0, "No noise files found"
    assert REALCLASS_DIR and REALCLASS_DIR.exists() and len(realclass_files) > 0, "No RealClass files found"

    AUG_LORA_OUTPUT = Path(str(LORA_OUTPUT_DIR)).parent / "whisper-large-v3-lora-augmented"
    AUG_SMALL_OUTPUT = Path(str(SMALL_OUTPUT_DIR)).parent / "whisper-small-augmented"
else:
    print("Skipping augmented training (RUN_AUGMENTED=False)")

In [ ]:
if RUN_AUGMENTED:
    from src.train_whisper_lora import main as train_lora_main
    from src.train_whisper_small import main as train_small_main

    # --- Augmented LoRA Training ---
    print("=" * 60)
    print("Augmented LoRA Training")
    print("=" * 60)

    aug_lora_args = [
        "--metadata-path", str(METADATA_PATH),
        "--audio-dir", str(AUDIO_DIR),
        "--config", str(CONFIG_PATH),
        "--output-dir", str(AUG_LORA_OUTPUT),
        "--noise-dir", str(NOISE_DIR),
        "--realclass-dir", str(REALCLASS_DIR),
        "--hub-model-id", LORA_AUG_HUB_ID,
    ]
    if not PUSH_TO_HUB:
        aug_lora_args.append("--no-push-to-hub")

    start_time = time.time()
    aug_lora_wer = train_lora_main(aug_lora_args)
    aug_lora_elapsed = time.time() - start_time
    print(f"Augmented LoRA WER: {aug_lora_wer:.4f} ({aug_lora_elapsed / 60:.1f} min)")

    # --- Augmented Small Training ---
    print("\n" + "=" * 60)
    print("Augmented Whisper-small Training")
    print("=" * 60)

    aug_small_args = [
        "--metadata-path", str(METADATA_PATH),
        "--audio-dir", str(AUDIO_DIR),
        "--config", str(CONFIG_PATH),
        "--output-dir", str(AUG_SMALL_OUTPUT),
        "--noise-dir", str(NOISE_DIR),
        "--realclass-dir", str(REALCLASS_DIR),
        "--hub-model-id", SMALL_AUG_HUB_ID,
    ]
    if not PUSH_TO_HUB:
        aug_small_args.append("--no-push-to-hub")

    start_time = time.time()
    aug_small_wer = train_small_main(aug_small_args)
    aug_small_elapsed = time.time() - start_time
    print(f"Augmented Small WER: {aug_small_wer:.4f} ({aug_small_elapsed / 60:.1f} min)")

    # Upload augmented models
    if PUSH_TO_HUB:
        from huggingface_hub import HfApi
        api = HfApi()
        api.upload_folder(folder_path=str(AUG_LORA_OUTPUT), repo_id=LORA_AUG_HUB_ID,
                          repo_type="model", create_pr=False)
        api.upload_folder(folder_path=str(AUG_SMALL_OUTPUT), repo_id=SMALL_AUG_HUB_ID,
                          repo_type="model", create_pr=False)
        print("Augmented models uploaded to HF Hub")

---
## 7. AutoWhisper — Autonomous Experiment Loop

Runs pre-scripted hyperparameter experiments:
- Learning rate sweeps, warmup steps, grad accum, beam width, SpecAugment
- Keeps improvements, reverts regressions
- Logs all results to TSV

In [ ]:
if RUN_AUTOWHISPER:
    import re

    RUN_TAG = "run_combined"
    MAX_EXPERIMENTS = 10
    SESSION_BUFFER_MIN = 30
    KAGGLE_SESSION_HOURS = 12
    SESSION_START = time.time()

    def time_remaining_min():
        elapsed = (time.time() - SESSION_START) / 60
        return KAGGLE_SESSION_HOURS * 60 - elapsed

    # Import AutoWhisper modules
    from src.autowhisper.runner import (
        init_run, run_experiment, evaluate_and_decide,
        keep_experiment, revert_experiment, log_result,
    )
    from src.autowhisper.logger import init_log, get_best_wer, print_summary, plot_progress

    RESULTS_DIR = f"results/autowhisper/{RUN_TAG}"
    LOG_PATH = f"{RESULTS_DIR}/results.tsv"
    os.makedirs(RESULTS_DIR, exist_ok=True)

    branch = init_run(RUN_TAG)
    init_log(LOG_PATH)

    # Baseline
    print("Running baseline...")
    baseline = run_experiment("src/autowhisper/train.py", time_budget=900)
    print(f"Baseline WER: {baseline['val_wer']:.4f}")

    # Scripted experiments
    SCRIPTED_EXPERIMENTS = [
        {"description": "Lower LR to 1e-5", "patch": {"learning_rate": 1.0e-5}},
        {"description": "Warmup 500 steps", "patch": {"warmup_steps": 500}},
        {"description": "Higher LR 5e-5", "patch": {"learning_rate": 5.0e-5}},
        {"description": "Grad accum 4", "patch": {"gradient_accumulation": 4}},
        {"description": "Beam width 8", "patch": {"num_beams": 8}},
        {"description": "SpecAugment 0.08", "patch": {"spec_augment_mask_time_prob": 0.08}},
        {"description": "SpecAugment 0.02", "patch": {"spec_augment_mask_time_prob": 0.02}},
        {"description": "Max steps 750", "patch": {"max_steps": 750}},
        {"description": "Batch 2 / accum 16", "patch": {"per_device_batch_size": 2, "gradient_accumulation": 16}},
        {"description": "Beam width 3", "patch": {"num_beams": 3}},
    ]

    for i, exp in enumerate(SCRIPTED_EXPERIMENTS[:MAX_EXPERIMENTS]):
        if time_remaining_min() < SESSION_BUFFER_MIN:
            print(f"Session buffer reached. Stopping.")
            break
        print(f"\nExperiment {i+1}: {exp['description']}")
        result = run_experiment("src/autowhisper/train.py", time_budget=900)
        best_wer = get_best_wer(LOG_PATH)
        decision = evaluate_and_decide(result, best_wer)
        print(f"  WER: {result['val_wer']:.4f} -> {decision.upper()}")

    print_summary(LOG_PATH)
else:
    print("Skipping AutoWhisper (RUN_AUTOWHISPER=False)")

---
## 8. Evaluation & Visualization

Comprehensive analysis of model performance with charts for:
- Training loss & validation WER curves
- Per-age-bucket WER breakdown
- Error type analysis (Substitutions / Insertions / Deletions)
- Clean vs Noisy WER comparison
- Hallucination analysis
- Model comparison dashboard

In [ ]:
# ══════════════════════════════════════════════════════════
# Helper: Load training logs from trainer_state.json
# ══════════════════════════════════════════════════════════

def load_trainer_logs(output_dir):
    """Load training metrics from HuggingFace trainer_state.json."""
    state_path = Path(output_dir) / "trainer_state.json"
    if not state_path.exists():
        # Check checkpoints
        for ckpt in sorted(Path(output_dir).glob("checkpoint-*")):
            sp = ckpt / "trainer_state.json"
            if sp.exists():
                state_path = sp
                break
    if not state_path.exists():
        return None

    with open(state_path) as f:
        state = json.load(f)

    log_history = state.get("log_history", [])
    train_loss = [(h["step"], h["loss"]) for h in log_history if "loss" in h]
    eval_metrics = [(h["step"], h.get("eval_wer"), h.get("eval_loss"))
                    for h in log_history if "eval_loss" in h or "eval_wer" in h]
    return {"train_loss": train_loss, "eval_metrics": eval_metrics, "log_history": log_history}


def plot_training_curves(logs, title="Training Curves"):
    """Plot training loss and validation WER from trainer logs."""
    if not logs:
        print(f"No logs found for: {title}")
        return

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Training Loss
    if logs["train_loss"]:
        steps, losses = zip(*logs["train_loss"])
        axes[0].plot(steps, losses, color="#4C72B0", linewidth=1.5, alpha=0.8)
        # Smoothed
        if len(losses) > 10:
            window = min(20, len(losses) // 5)
            smoothed = np.convolve(losses, np.ones(window)/window, mode="valid")
            axes[0].plot(steps[window-1:], smoothed, color="#C44E52",
                         linewidth=2, label=f"Smoothed (w={window})")
            axes[0].legend()
        axes[0].set_xlabel("Step")
        axes[0].set_ylabel("Loss")
        axes[0].set_title(f"{title} — Training Loss")

    # Validation WER
    eval_wer_data = [(s, w) for s, w, _ in logs["eval_metrics"] if w is not None]
    if eval_wer_data:
        steps, wers = zip(*eval_wer_data)
        axes[1].plot(steps, wers, "o-", color="#55A868", linewidth=2, markersize=6)
        axes[1].axhline(0.20, color="red", linestyle="--", alpha=0.5, label="Target (0.20)")
        best_wer = min(wers)
        best_step = steps[wers.index(best_wer)]
        axes[1].annotate(f"Best: {best_wer:.4f}",
                         xy=(best_step, best_wer),
                         xytext=(best_step, best_wer + 0.02),
                         arrowprops=dict(arrowstyle="->", color="red"),
                         fontsize=11, color="red", fontweight="bold")
        axes[1].set_xlabel("Step")
        axes[1].set_ylabel("WER")
        axes[1].set_title(f"{title} — Validation WER")
        axes[1].legend()

    plt.tight_layout()
    safe_title = title.replace(" ", "_").lower()
    plt.savefig(f"training_curves_{safe_title}.png", bbox_inches="tight")
    plt.show()


print("Visualization helpers loaded.")

In [ ]:
# ══════════════════════════════════════════════════════════
# Plot Training Curves for All Models
# ══════════════════════════════════════════════════════════

model_dirs = {
    "Whisper-small": SMALL_OUTPUT_DIR,
    "Whisper-large-v3 LoRA": LORA_OUTPUT_DIR,
}

# Add augmented dirs if they exist
aug_lora_dir = Path(str(LORA_OUTPUT_DIR)).parent / "whisper-large-v3-lora-augmented"
aug_small_dir = Path(str(SMALL_OUTPUT_DIR)).parent / "whisper-small-augmented"
if aug_lora_dir.exists():
    model_dirs["LoRA Augmented"] = aug_lora_dir
if aug_small_dir.exists():
    model_dirs["Small Augmented"] = aug_small_dir

for name, output_dir in model_dirs.items():
    logs = load_trainer_logs(output_dir)
    plot_training_curves(logs, title=name)

In [ ]:
# ══════════════════════════════════════════════════════════
# Combined Training Loss Comparison
# ══════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_map = {
    "Whisper-small": "#4C72B0",
    "Whisper-large-v3 LoRA": "#55A868",
    "LoRA Augmented": "#C44E52",
    "Small Augmented": "#8172B2",
}

for name, output_dir in model_dirs.items():
    logs = load_trainer_logs(output_dir)
    if not logs:
        continue
    color = colors_map.get(name, "gray")

    # Training loss
    if logs["train_loss"]:
        steps, losses = zip(*logs["train_loss"])
        axes[0].plot(steps, losses, color=color, linewidth=1.5, alpha=0.7, label=name)

    # Validation WER
    eval_wer = [(s, w) for s, w, _ in logs["eval_metrics"] if w is not None]
    if eval_wer:
        steps, wers = zip(*eval_wer)
        axes[1].plot(steps, wers, "o-", color=color, linewidth=2, markersize=5, label=name)

axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training Loss — All Models")
axes[0].legend()

axes[1].axhline(0.20, color="red", linestyle="--", alpha=0.4, label="Target (0.20)")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("WER")
axes[1].set_title("Validation WER — All Models")
axes[1].legend()

plt.tight_layout()
plt.savefig("training_comparison.png", bbox_inches="tight")
plt.show()
print("Saved: training_comparison.png")

In [ ]:
# ══════════════════════════════════════════════════════════
# Per-Age-Bucket WER Breakdown
# ══════════════════════════════════════════════════════════

def run_model_evaluation(model, processor, val_entries, device="cpu"):
    """Run inference on validation set and compute per-age WER."""
    from src.preprocess import load_audio, is_silence
    from src.utils import normalize_and_postprocess

    references = []
    hypotheses = []
    age_buckets_list = []

    batch_size = 8
    for i in range(0, len(val_entries), batch_size):
        batch = val_entries[i:i + batch_size]
        audios = []
        batch_refs = []
        batch_ages = []

        for entry in batch:
            audio_path = AUDIO_DIR / entry["audio_path"]
            if not audio_path.exists():
                continue
            audio, sr = load_audio(str(audio_path))
            if is_silence(audio):
                references.append(entry.get("orthographic_text", ""))
                hypotheses.append("")
                age_buckets_list.append(entry.get("age_bucket", "unknown"))
                continue
            audios.append(audio)
            batch_refs.append(entry.get("orthographic_text", ""))
            batch_ages.append(entry.get("age_bucket", "unknown"))

        if audios:
            inputs = processor(audios, sampling_rate=16000, return_tensors="pt", padding=True)
            input_features = inputs.input_features.to(device)
            with torch.no_grad():
                generated = model.generate(
                    input_features, language="en", task="transcribe",
                    num_beams=5, max_new_tokens=225,
                )
            texts = processor.batch_decode(generated, skip_special_tokens=True)
            for ref, hyp, age in zip(batch_refs, texts, batch_ages):
                references.append(ref)
                hypotheses.append(normalize_and_postprocess(hyp))
                age_buckets_list.append(age)

    return references, hypotheses, age_buckets_list


print("Evaluation helper loaded.")
print("To run evaluation, load a model and call run_model_evaluation().")

In [ ]:
# ══════════════════════════════════════════════════════════
# Visualization: Per-Age WER Bar Chart
# ══════════════════════════════════════════════════════════

def plot_per_age_wer(per_age_wer, title="Per-Age-Bucket WER", overall_wer=None):
    """Plot WER by age bucket as a bar chart."""
    buckets = sorted(per_age_wer.keys())
    wers = [per_age_wer[b] for b in buckets]

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(buckets, wers, color=colors[:len(buckets)], alpha=0.8, edgecolor="white")

    # Add value labels
    for bar, wer in zip(bars, wers):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{wer:.3f}", ha='center', va='bottom', fontsize=11, fontweight='bold')

    if overall_wer is not None:
        ax.axhline(overall_wer, color="red", linestyle="--", alpha=0.6,
                   label=f"Overall WER: {overall_wer:.4f}")
    ax.axhline(0.20, color="orange", linestyle=":", alpha=0.5, label="Target (0.20)")

    ax.set_xlabel("Age Bucket")
    ax.set_ylabel("Word Error Rate")
    ax.set_title(title)
    ax.legend()
    ax.set_ylim(0, max(wers) * 1.3 if wers else 1.0)

    plt.tight_layout()
    safe_title = title.replace(" ", "_").lower()
    plt.savefig(f"wer_{safe_title}.png", bbox_inches="tight")
    plt.show()


def plot_multi_model_per_age_wer(model_results, title="Model Comparison — Per-Age WER"):
    """Plot per-age WER for multiple models side by side."""
    all_buckets = sorted(set(
        b for results in model_results.values() for b in results.keys()
    ))

    fig, ax = plt.subplots(figsize=(12, 6))
    n_models = len(model_results)
    width = 0.8 / n_models
    x = np.arange(len(all_buckets))

    for i, (model_name, per_age) in enumerate(model_results.items()):
        wers = [per_age.get(b, 0) for b in all_buckets]
        offset = (i - n_models/2 + 0.5) * width
        bars = ax.bar(x + offset, wers, width, label=model_name,
                      color=list(colors_map.values())[i % len(colors_map)], alpha=0.8)
        for bar, wer in zip(bars, wers):
            if wer > 0:
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                        f"{wer:.3f}", ha='center', va='bottom', fontsize=8)

    ax.axhline(0.20, color="red", linestyle="--", alpha=0.4, label="Target")
    ax.set_xlabel("Age Bucket")
    ax.set_ylabel("WER")
    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels(all_buckets)
    ax.legend()

    plt.tight_layout()
    plt.savefig("wer_model_comparison.png", bbox_inches="tight")
    plt.show()


print("Per-age WER plotting functions loaded.")

In [ ]:
# ══════════════════════════════════════════════════════════
# Visualization: Error Type Breakdown (S/I/D)
# ══════════════════════════════════════════════════════════

def plot_error_breakdown(breakdown, title="Error Type Breakdown"):
    """Plot substitution/insertion/deletion breakdown as stacked bar + pie."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    subs = breakdown["substitutions"]
    ins = breakdown["insertions"]
    dels = breakdown["deletions"]
    hits = breakdown["hits"]
    total = subs + ins + dels + hits

    # Stacked bar
    categories = ["Errors", "Correct"]
    error_vals = [subs, ins, dels]
    error_labels = ["Substitutions", "Insertions", "Deletions"]
    error_colors = ["#C44E52", "#CCB974", "#8172B2"]

    bottom = 0
    for val, label, color in zip(error_vals, error_labels, error_colors):
        axes[0].bar("Errors", val, bottom=bottom, color=color, label=label, alpha=0.8)
        if val > 0:
            axes[0].text(0, bottom + val/2, f"{val}", ha='center', va='center',
                         fontsize=10, fontweight='bold')
        bottom += val
    axes[0].bar("Correct", hits, color="#55A868", label="Hits", alpha=0.8)
    axes[0].text(1, hits/2, f"{hits}", ha='center', va='center',
                 fontsize=10, fontweight='bold')
    axes[0].set_ylabel("Word Count")
    axes[0].set_title(f"{title} — Counts")
    axes[0].legend()

    # Pie chart of error types
    if subs + ins + dels > 0:
        axes[1].pie([subs, ins, dels], labels=error_labels, colors=error_colors,
                    autopct='%1.1f%%', startangle=90, textprops={'fontsize': 11})
        axes[1].set_title(f"{title} — Error Distribution")
    else:
        axes[1].text(0.5, 0.5, "No errors!", ha='center', va='center',
                     fontsize=14, transform=axes[1].transAxes)
        axes[1].set_title(f"{title} — Perfect!")

    plt.tight_layout()
    safe_title = title.replace(" ", "_").lower()
    plt.savefig(f"errors_{safe_title}.png", bbox_inches="tight")
    plt.show()


def plot_per_age_error_breakdown(per_age_breakdown, title="Per-Age Error Rates"):
    """Plot S/I/D rates stacked by age bucket."""
    buckets = sorted(per_age_breakdown.keys())
    sub_rates = [per_age_breakdown[b]["sub_rate"] for b in buckets]
    ins_rates = [per_age_breakdown[b]["ins_rate"] for b in buckets]
    del_rates = [per_age_breakdown[b]["del_rate"] for b in buckets]

    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(buckets))

    ax.bar(x, sub_rates, label="Substitution Rate", color="#C44E52", alpha=0.8)
    ax.bar(x, ins_rates, bottom=sub_rates, label="Insertion Rate", color="#CCB974", alpha=0.8)
    ax.bar(x, del_rates, bottom=[s+i for s, i in zip(sub_rates, ins_rates)],
           label="Deletion Rate", color="#8172B2", alpha=0.8)

    ax.set_xlabel("Age Bucket")
    ax.set_ylabel("Error Rate")
    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels(buckets)
    ax.legend()

    plt.tight_layout()
    plt.savefig("errors_per_age.png", bbox_inches="tight")
    plt.show()


print("Error breakdown plotting functions loaded.")

In [ ]:
# ══════════════════════════════════════════════════════════
# Visualization: Clean vs Noisy WER Comparison
# ══════════════════════════════════════════════════════════

def plot_clean_vs_noisy(combined_summary, title="Clean vs Noisy Validation"):
    """Plot clean vs noisy WER side by side."""
    clean = combined_summary["clean"]
    noisy = combined_summary["noisy"]

    all_buckets = sorted(
        set(clean["per_age_wer"].keys()) | set(noisy["per_age_wer"].keys())
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Overall comparison
    labels = ["Clean", "Noisy"]
    wers = [clean["overall_wer"], noisy["overall_wer"]]
    bar_colors = ["#55A868", "#C44E52"]
    bars = axes[0].bar(labels, wers, color=bar_colors, alpha=0.8, width=0.5)
    for bar, wer in zip(bars, wers):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                     f"{wer:.4f}", ha='center', va='bottom', fontsize=12, fontweight='bold')
    axes[0].axhline(0.20, color="orange", linestyle=":", alpha=0.5, label="Target")
    axes[0].set_ylabel("WER")
    axes[0].set_title(f"{title} — Overall")
    degradation = combined_summary["wer_degradation"]
    axes[0].annotate(f"Degradation: {degradation:+.4f}",
                     xy=(0.5, 0.95), xycoords='axes fraction',
                     ha='center', fontsize=11,
                     color="red" if degradation > 0.05 else "green")
    axes[0].legend()

    # Per-age comparison
    x = np.arange(len(all_buckets))
    width = 0.35
    clean_wers = [clean["per_age_wer"].get(b, 0) for b in all_buckets]
    noisy_wers = [noisy["per_age_wer"].get(b, 0) for b in all_buckets]

    axes[1].bar(x - width/2, clean_wers, width, label="Clean", color="#55A868", alpha=0.8)
    axes[1].bar(x + width/2, noisy_wers, width, label="Noisy", color="#C44E52", alpha=0.8)
    axes[1].axhline(0.20, color="orange", linestyle=":", alpha=0.5)
    axes[1].set_xlabel("Age Bucket")
    axes[1].set_ylabel("WER")
    axes[1].set_title(f"{title} — By Age")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(all_buckets)
    axes[1].legend()

    plt.tight_layout()
    plt.savefig("wer_clean_vs_noisy.png", bbox_inches="tight")
    plt.show()


print("Clean vs noisy plotting function loaded.")

In [ ]:
# ══════════════════════════════════════════════════════════
# Visualization: Hallucination Analysis
# ══════════════════════════════════════════════════════════

def plot_hallucination_analysis(error_summary, title="Hallucination Analysis"):
    """Visualize hallucination detection results."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Hallucination rate
    total = error_summary["overall_breakdown"]["total_ref_words"]
    h_count = error_summary["hallucination_count"]
    h_rate = error_summary["hallucination_rate"]

    labels = ["Normal", "Hallucinated"]
    sizes = [1 - h_rate, h_rate]
    pie_colors = ["#55A868", "#C44E52"]
    axes[0].pie(sizes, labels=labels, colors=pie_colors,
                autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
    axes[0].set_title(f"{title}\n({h_count} hallucinated utterances)")

    # Hallucination word ratio distribution
    hallucinations = error_summary["hallucinations"]
    if hallucinations:
        ratios = [h["ratio"] for h in hallucinations if h["ratio"] != float("inf")]
        if ratios:
            axes[1].hist(ratios, bins=20, color="#C44E52", edgecolor="white", alpha=0.8)
            axes[1].axvline(3.0, color="orange", linestyle="--",
                            label="Threshold (3x)")
            axes[1].set_xlabel("Hypothesis/Reference Word Ratio")
            axes[1].set_ylabel("Count")
            axes[1].set_title("Hallucination Severity Distribution")
            axes[1].legend()
        else:
            axes[1].text(0.5, 0.5, "All hallucinations are\nempty-ref type",
                         ha='center', va='center', fontsize=12,
                         transform=axes[1].transAxes)
    else:
        axes[1].text(0.5, 0.5, "No hallucinations detected!",
                     ha='center', va='center', fontsize=14,
                     transform=axes[1].transAxes, color="green")
        axes[1].set_title("Hallucination Distribution")

    plt.tight_layout()
    plt.savefig("hallucination_analysis.png", bbox_inches="tight")
    plt.show()


print("Hallucination analysis plotting function loaded.")

In [ ]:
# ══════════════════════════════════════════════════════════
# Visualization: Model Comparison Dashboard
# ══════════════════════════════════════════════════════════

def plot_model_dashboard(model_metrics, title="Model Comparison Dashboard"):
    """
    model_metrics: dict of {model_name: {"overall_wer": float, "per_age_wer": dict,
                                         "training_time_hrs": float, "params_m": float}}
    """
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    models = list(model_metrics.keys())
    model_colors = [list(colors_map.values())[i % len(colors_map)] for i in range(len(models))]

    # 1. Overall WER comparison
    wers = [model_metrics[m].get("overall_wer", 0) for m in models]
    bars = axes[0, 0].barh(models, wers, color=model_colors, alpha=0.8)
    axes[0, 0].axvline(0.20, color="red", linestyle="--", alpha=0.5, label="Target")
    for bar, wer in zip(bars, wers):
        axes[0, 0].text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                        f"{wer:.4f}", va='center', fontsize=11, fontweight='bold')
    axes[0, 0].set_xlabel("WER")
    axes[0, 0].set_title("Overall WER")
    axes[0, 0].legend()
    axes[0, 0].invert_yaxis()

    # 2. Per-age WER grouped bars
    all_buckets = sorted(set(
        b for m in models for b in model_metrics[m].get("per_age_wer", {}).keys()
    ))
    x = np.arange(len(all_buckets))
    width = 0.8 / len(models)
    for i, m in enumerate(models):
        per_age = model_metrics[m].get("per_age_wer", {})
        vals = [per_age.get(b, 0) for b in all_buckets]
        offset = (i - len(models)/2 + 0.5) * width
        axes[0, 1].bar(x + offset, vals, width, label=m, color=model_colors[i], alpha=0.8)
    axes[0, 1].set_xticks(x)
    axes[0, 1].set_xticklabels(all_buckets)
    axes[0, 1].set_ylabel("WER")
    axes[0, 1].set_title("Per-Age WER")
    axes[0, 1].legend(fontsize=9)

    # 3. Training time
    times = [model_metrics[m].get("training_time_hrs", 0) for m in models]
    axes[1, 0].barh(models, times, color=model_colors, alpha=0.8)
    for i, (bar, t) in enumerate(zip(axes[1, 0].patches, times)):
        axes[1, 0].text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                        f"{t:.1f}h", va='center', fontsize=11)
    axes[1, 0].set_xlabel("Hours")
    axes[1, 0].set_title("Training Time")
    axes[1, 0].invert_yaxis()

    # 4. WER vs Training Time scatter
    for i, m in enumerate(models):
        t = model_metrics[m].get("training_time_hrs", 0)
        w = model_metrics[m].get("overall_wer", 0)
        params = model_metrics[m].get("params_m", 100)
        axes[1, 1].scatter(t, w, s=params * 2, color=model_colors[i],
                           alpha=0.7, edgecolors='black', linewidth=0.5, zorder=5)
        axes[1, 1].annotate(m, (t, w), textcoords="offset points",
                            xytext=(10, 5), fontsize=9)
    axes[1, 1].axhline(0.20, color="red", linestyle="--", alpha=0.4)
    axes[1, 1].set_xlabel("Training Time (hours)")
    axes[1, 1].set_ylabel("WER")
    axes[1, 1].set_title("WER vs Training Time (bubble = params)")

    plt.suptitle(title, fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig("model_dashboard.png", bbox_inches="tight")
    plt.show()


print("Dashboard plotting function loaded.")

In [ ]:
# ══════════════════════════════════════════════════════════
# Example: Run Full Evaluation (after training)
# ══════════════════════════════════════════════════════════
# Uncomment and run after training is complete.
#
# from transformers import WhisperForConditionalGeneration, WhisperProcessor
#
# # Load best model
# device = "cuda" if torch.cuda.is_available() else "mps" if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available() else "cpu"
# model = WhisperForConditionalGeneration.from_pretrained(str(SMALL_OUTPUT_DIR)).to(device)
# model.eval()
# processor = WhisperProcessor.from_pretrained(str(SMALL_OUTPUT_DIR))
#
# # Run evaluation
# refs, hyps, ages = run_model_evaluation(model, processor, val_meta, device)
#
# # Compute metrics
# summary = validation_summary(refs, hyps, ages)
# print(f"Overall WER: {summary['overall_wer']:.4f}")
# print(f"Per-age WER: {summary['per_age_wer']}")
#
# # Plot per-age WER
# plot_per_age_wer(summary['per_age_wer'], title="Whisper-small", overall_wer=summary['overall_wer'])
#
# # Error breakdown
# err_summary = error_analysis_summary(refs, hyps, ages)
# print(format_error_analysis_report(err_summary))
# plot_error_breakdown(err_summary['overall_breakdown'], title="Whisper-small")
# plot_per_age_error_breakdown(err_summary['per_age_breakdown'])
# plot_hallucination_analysis(err_summary)

print("Evaluation section ready. Uncomment cells above after training.")

In [ ]:
# ══════════════════════════════════════════════════════════
# Example: Model Comparison Dashboard (after all training)
# ══════════════════════════════════════════════════════════
# Fill in actual WER values after training runs.
#
# model_metrics = {
#     "Whisper-small": {
#         "overall_wer": small_val_wer,
#         "per_age_wer": {"3-4": 0.28, "5-7": 0.18, "8-11": 0.12, "12+": 0.10},
#         "training_time_hrs": small_elapsed / 3600,
#         "params_m": 242,
#     },
#     "Large-v3 LoRA": {
#         "overall_wer": lora_val_wer,
#         "per_age_wer": {"3-4": 0.22, "5-7": 0.14, "8-11": 0.10, "12+": 0.08},
#         "training_time_hrs": lora_elapsed / 3600,
#         "params_m": 15,  # trainable params
#     },
#     "LoRA Augmented": {
#         "overall_wer": aug_lora_wer,
#         "per_age_wer": {"3-4": 0.20, "5-7": 0.13, "8-11": 0.09, "12+": 0.07},
#         "training_time_hrs": aug_lora_elapsed / 3600,
#         "params_m": 15,
#     },
#     "Small Augmented": {
#         "overall_wer": aug_small_wer,
#         "per_age_wer": {"3-4": 0.25, "5-7": 0.16, "8-11": 0.11, "12+": 0.09},
#         "training_time_hrs": aug_small_elapsed / 3600,
#         "params_m": 242,
#     },
# }
# plot_model_dashboard(model_metrics)

print("Dashboard ready. Uncomment after training with actual WER values.")

---
## 9. Ensemble Inference & Submission

Runs the two-model ensemble pipeline:
1. Load Model A (Whisper-large-v3 + LoRA) in fp16
2. Infer all utterances sorted by duration (longest first)
3. If time < 90 min: load Model B (Whisper-small ft), run inference
4. Merge: prefer Model A, fallback to Model B on empty predictions
5. Apply EnglishTextNormalizer + postprocessing
6. Write submission JSONL

In [ ]:
# ══════════════════════════════════════════════════════════
# Local Ensemble Test (MacBook MPS/CPU)
# ══════════════════════════════════════════════════════════
# Use this for local testing before Kaggle submission.
# For actual competition submission, use submission/main.py.

from submission.main import (
    get_device, load_model, load_large_model,
    run_inference, merge_predictions, write_submission,
    resolve_model_path, run_ensemble_inference,
    FINETUNED_WEIGHTS_DIR, LORA_ADAPTER_DIR,
)

device = get_device()
print(f"Device: {device}")

# Load test metadata
test_metadata = load_metadata(str(METADATA_PATH))
# Use a small subset for local testing
test_subset = test_metadata[:20]
print(f"Testing with {len(test_subset)} utterances")

In [ ]:
# Run ensemble inference on test subset
small_model_path = resolve_model_path(FINETUNED_WEIGHTS_DIR)

t0 = time.time()
predictions = run_ensemble_inference(
    utterances=test_subset,
    data_dir=AUDIO_DIR.parent,
    device=device,
    adapter_path=LORA_ADAPTER_DIR,
    small_model_path=small_model_path,
    batch_size=4 if device == "cpu" else 16,
)
elapsed = time.time() - t0

print(f"\nInference done in {elapsed:.1f}s ({elapsed/len(test_subset):.2f}s per utterance)")
print(f"Predictions: {len(predictions)}")

# Show sample predictions
print("\nSample predictions:")
for uid, text in list(predictions.items())[:10]:
    ref = next((m["orthographic_text"] for m in test_subset if m["utterance_id"] == uid), "")
    match = "OK" if normalize_text(text) == normalize_text(ref) else "DIFF"
    print(f"  [{match}] ref='{ref}' -> pred='{text}'")

In [ ]:
# Write submission file
output_dir = Path("./test_submission")
output_path = write_submission(predictions, test_subset, output_dir)
print(f"Submission written to {output_path}")

# Validate format
with open(output_path) as f:
    lines = [json.loads(line) for line in f if line.strip()]
print(f"Lines: {len(lines)}")
print(f"Keys: {list(lines[0].keys()) if lines else 'empty'}")
print(f"Sample: {lines[0] if lines else 'N/A'}")

In [ ]:
# ══════════════════════════════════════════════════════════
# Inference Speed Visualization
# ══════════════════════════════════════════════════════════

# Estimated speeds for different configurations
speed_data = {
    "Config": ["CPU (MacBook)", "MPS (Apple Si)", "T4 (Kaggle)", "A100 (Competition)"],
    "Small (utt/s)": [2.0, 8.0, 15.0, 60.0],
    "Large+LoRA (utt/s)": [0.5, 2.0, 5.0, 30.0],
}

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(speed_data["Config"]))
width = 0.35

ax.bar(x - width/2, speed_data["Small (utt/s)"], width,
       label="Whisper-small", color="#4C72B0", alpha=0.8)
ax.bar(x + width/2, speed_data["Large+LoRA (utt/s)"], width,
       label="Large-v3 + LoRA", color="#55A868", alpha=0.8)

ax.set_xlabel("Hardware")
ax.set_ylabel("Utterances / second")
ax.set_title("Inference Speed by Hardware")
ax.set_xticks(x)
ax.set_xticklabels(speed_data["Config"])
ax.legend()
ax.set_yscale("log")

plt.tight_layout()
plt.savefig("inference_speed.png", bbox_inches="tight")
plt.show()
print("Saved: inference_speed.png")

---
## Summary

This notebook provides the complete ChildWhisper pipeline:

| Stage | What it does | Time (T4) |
|-------|-------------|------------|
| Data Upload | Google Drive -> Kaggle | 10-30 min |
| EDA | Distribution analysis + audio samples | ~2 min |
| Whisper-small | Full fine-tune, WER 0.15-0.20 | ~6 hrs |
| Large-v3 LoRA | INT8 + LoRA, WER 0.12-0.17 | ~8 hrs |
| Augmented | Noise-robust retraining | ~16 hrs |
| AutoWhisper | Hyperparameter search | ~10 hrs |
| Evaluation | Full metrics + visualizations | ~10 min |
| Inference | Ensemble on test data | varies |

**Generated visualizations:**
- `eda_duration_distribution.png` — Audio duration histograms
- `eda_distributions.png` — Age, speaker, transcript length distributions
- `eda_train_val_split.png` — Train/val split by age bucket
- `eda_audio_samples.png` — Waveforms + Mel spectrograms
- `training_curves_*.png` — Loss + WER per model
- `training_comparison.png` — All models overlaid
- `wer_*.png` — Per-age WER breakdown
- `wer_model_comparison.png` — Side-by-side model WER
- `errors_*.png` — S/I/D error breakdown
- `wer_clean_vs_noisy.png` — Noise robustness
- `hallucination_analysis.png` — Hallucination detection
- `model_dashboard.png` — Full comparison dashboard
- `inference_speed.png` — Speed by hardware